# LeWM Two-Room — feasibility probe

**This notebook does not train anything.** It answers four questions, in order, and stops. Each answer replaces a guess with a measurement.

1. **What's actually inside the dataset file?** (We inspect it rather than    assume key names — this is where loaders silently go wrong.)
2. **Do the frames and actions line up?** (If they're off by one step,    everything downstream trains on nonsense and still looks plausible.)
3. **What batch size fits in a T4's 16 GB?** (The reference uses 128. If we    can't fit that, it's a *deviation*, not just a slowdown — see the final cell.)
4. **How many hours would a full run take?** (Measured, then multiplied out.)

Run the cells top to bottom. The last cell prints the verdict.

> **Reference facts this probe checks against** (from the LeWM repo config and paper):
> 10,000 episodes, ~92 steps each, frameskip 5, image size 224, embed dim 192,
> history size 3, batch size 128, AdamW lr 5e-5, seed 3072.
> Note: repo config says `max_epochs: 100`; paper Appendix E says Two-Room trains
> for **10 epochs**. We compute both — the difference is 10x your compute bill.

## 0 — Setup

Confirm we actually have a GPU, and see which one. Colab hands out different cards; the answer changes the timing.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  memory: {props.total_memory/1e9:.1f} GB')
else:
    print('NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun.')

Wed Jul 15 14:00:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Mount Drive. The dataset is 3.43 GB — download it ONCE to Drive, never again.
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/lewm_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('data dir:', DATA_DIR)

Mounted at /content/drive
data dir: /content/drive/MyDrive/lewm_data


In [ ]:
!pip install -q huggingface_hub h5py einops transformers 2>&1 | tail -1
print('installed')

installed


## 1 — Get the dataset

`quentinll/lewm-tworooms` on HuggingFace, 3.43 GB total. The repo says the files are compressed and need decompressing after download.

We first **list** what's in the repo before downloading — we don't yet know the exact filenames, and guessing wastes a 3.43 GB download.

In [ ]:
from huggingface_hub import list_repo_files

REPO = 'quentinll/lewm-tworooms'
files = list_repo_files(REPO, repo_type='dataset')
print(f'{len(files)} files in {REPO}:')
for f in files:
    print('  ', f)

3 files in quentinll/lewm-tworooms:
   .gitattributes
   README.md
   tworoom.tar.zst


In [ ]:
# Download the dataset. From the listing, the only data file is tworoom.tar.zst
from huggingface_hub import hf_hub_download
import os

TARGET = "tworoom.tar.zst"   # quotes matter — this is text, not code

path = hf_hub_download(REPO, TARGET, repo_type="dataset", local_dir=DATA_DIR)
print("downloaded to:", path)
print("size: %.2f GB" % (os.path.getsize(path) / 1e9))

tworoom.tar.zst:   0%|          | 0.00/3.43G [00:00<?, ?B/s]

downloaded to: /content/drive/MyDrive/lewm_data/tworoom.tar.zst
size: 3.43 GB


In [ ]:
# tworoom.tar.zst is zstd-compressed, not gzip — Colab doesn't ship zstd by default.
!apt-get install -qq zstd 2>&1 | tail -1

# Extract. Expect this to take a few minutes and to roughly double disk use.
!tar --use-compress-program=unzstd -xf "{path}" -C "{DATA_DIR}"

# Find whatever .h5 file came out — don't assume the name.
import glob
h5_files = glob.glob(os.path.join(DATA_DIR, "**", "*.h5"), recursive=True)
print("extracted .h5 files:")
for f in h5_files:
    print(f"   {f}   ({os.path.getsize(f)/1e9:.2f} GB)")

H5_PATH = h5_files[0] if h5_files else None
print()
print("H5_PATH =", H5_PATH)

Processing triggers for man-db (2.10.2-1) ...
extracted .h5 files:
   /content/drive/MyDrive/lewm_data/tworoom.h5   (12.78 GB)

H5_PATH = /content/drive/MyDrive/lewm_data/tworoom.h5


## 2 — What's actually in the file?

**This is the cell that matters most.** We print the real structure instead of assuming it. The training config asks for three things — `pixels`, `action`, `proprio` — so we're checking those exist and finding their shapes and types.

What we're looking for:
- how episodes are stored (one big array? one group per episode?)
- the shape of the frames (how many, what size, colour channels where?)
- the shape of the actions (how many numbers per action?)
- the number range of the pixels (0-255 whole numbers, or 0-1 decimals?)

In [ ]:
import h5py
import numpy as np

H5_PATH = "/content/drive/MyDrive/lewm_data/tworoom.h5"  # <-- set to the .h5 file path after download/decompress

if H5_PATH is None:
    raise SystemExit('Set H5_PATH to the extracted .h5 file.')

def describe(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f'  {name:40s} shape={str(obj.shape):28s} dtype={obj.dtype}')
    else:
        print(f'  {name:40s} [group]')

with h5py.File(H5_PATH, 'r') as f:
    print('=== top-level keys ===')
    print(' ', list(f.keys()))
    print()
    print('=== full structure (first 40 entries) ===')
    shown = []
    f.visititems(lambda n, o: shown.append((n, o)))
    for n, o in shown[:40]:
        describe(n, o)
    if len(shown) > 40:
        print(f'  ... and {len(shown)-40} more')
    print()
    print('=== attributes ===')
    for k, v in f.attrs.items():
        print(f'  {k}: {v}')

=== top-level keys ===
  ['action', 'distance_to_target', 'ep_idx', 'ep_len', 'ep_offset', 'id', 'observation', 'pixels', 'pos_agent', 'pos_target', 'proprio', 'render_time', 'reward', 'step_idx', 'terminated', 'truncated']

=== full structure (first 40 entries) ===
  action                                   shape=(920809, 2)                  dtype=float32
  distance_to_target                       shape=(920809,)                    dtype=float64
  ep_idx                                   shape=(920809,)                    dtype=int32
  ep_len                                   shape=(10000,)                     dtype=int32
  ep_offset                                shape=(10000,)                     dtype=int64
  id                                       shape=(920809,)                    dtype=int64
  observation                              shape=(920809, 10)                 dtype=float64
  pixels                                   shape=(920809, 224, 224, 3)        dtype=uint8
  pos_a

In [ ]:
# Pull one small sample and report ranges/types. Adjust the key paths to match
# whatever the structure above revealed.
with h5py.File(H5_PATH, 'r') as f:
    # EDIT these two lines to the real paths from the cell above:
    px = f['pixels']      # e.g. f['pixels'] or f['episode_0/pixels']
    ac = f['action']

    print('pixels :', px.shape, px.dtype)
    print('action :', ac.shape, ac.dtype)

    sample_px = px[0] if px.ndim >= 4 else px[:1]
    sample_ac = ac[0] if ac.ndim >= 2 else ac[:1]

    print()
    print('one frame  -> shape', np.asarray(sample_px).shape,
          ' min', np.asarray(sample_px).min(), ' max', np.asarray(sample_px).max())
    print('one action -> ', np.asarray(sample_ac))
    print()
    print('IMPORTANT: pixel max ~255 means whole-number colours (needs /255 later);')
    print('           pixel max ~1.0 means already scaled.')

## 3 — Do the frames and actions line up?

The single most common way a reproduction dies silently: frames and actions off by one step. Training still runs, loss still falls, results are garbage.

So we **look**. Plot a short run of frames in order, print the actions between them, and check by eye that the red dot moves the way the actions say. A rightward action should move the dot right, in the frame *after* the action.

In [ ]:
import matplotlib.pyplot as plt

N_SHOW = 8
with h5py.File(H5_PATH, 'r') as f:
    frames = np.asarray(f['pixels'][:N_SHOW])   # EDIT path if needed
    acts   = np.asarray(f['action'][:N_SHOW])

# Put colour channels last for plotting if they're first
if frames.ndim == 4 and frames.shape[1] in (1, 3):
    frames = frames.transpose(0, 2, 3, 1)

fig, axes = plt.subplots(1, N_SHOW, figsize=(2*N_SHOW, 2.6))
for i, ax in enumerate(axes):
    img = frames[i]
    ax.imshow(img.astype('uint8') if img.max() > 1.5 else img)
    ax.set_title(f't={i}', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

print('actions between these frames:')
for i in range(N_SHOW):
    print(f'  t={i} -> {np.round(acts[i], 3)}')
print()
print('CHECK BY EYE: does the dot move in the direction each action says,')
print('in the NEXT frame? If it moves BEFORE the action, you are off by one.')

## 4 — What batch size actually fits?

The reference config uses **batch 128**, and each item is a clip of 4 frames — so that's **512 images of 224x224 through the encoder per step**, plus the backward pass. A T4 has 16 GB. This cell finds the largest batch that fits by trying and catching failures.

We use a real ViT-Tiny at 224x224 — the same encoder LeWM uses — so the memory figure is honest rather than a guess.

In [ ]:
import torch, gc
from transformers import ViTModel, ViTConfig

HISTORY = 4      # frames per clip (history_size 3 + 1 target)
IMG = 224

def build_encoder():
    cfg = ViTConfig(hidden_size=192, num_hidden_layers=12, num_attention_heads=3,
                    intermediate_size=768, image_size=IMG, patch_size=14)
    return ViTModel(cfg).cuda()

def try_batch(bs, enc):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    try:
        x = torch.randn(bs*HISTORY, 3, IMG, IMG, device='cuda')
        with torch.autocast('cuda', dtype=torch.bfloat16):
            out = enc(x).last_hidden_state[:, 0]
            loss = out.square().mean()
        loss.backward()
        peak = torch.cuda.max_memory_allocated()/1e9
        del x, out, loss
        return True, peak
    except torch.cuda.OutOfMemoryError:
        return False, None

enc = build_encoder()
n_params = sum(p.numel() for p in enc.parameters())
print(f'encoder params: {n_params/1e6:.1f}M  (paper says ~5M for ViT-Tiny)')
print()

fits = {}
for bs in [8, 16, 32, 64, 128]:
    ok, peak = try_batch(bs, enc)
    fits[bs] = ok
    if ok:
        print(f'  batch {bs:3d}: FITS   (peak {peak:.1f} GB, {bs*HISTORY} images/step)')
    else:
        print(f'  batch {bs:3d}: OUT OF MEMORY')
    torch.cuda.empty_cache(); gc.collect()

MAX_BATCH = max([b for b, ok in fits.items() if ok], default=0)
print()
print(f'>>> largest batch that fits: {MAX_BATCH}  (reference uses 128)')

## 5 — How long is one step, really?

Time the encoder forward+backward at the largest batch that fits. This is the dominant cost — the predictor is small by comparison, so this gives a floor, not the full number. Treat the projection as optimistic.

In [ ]:
import time

BS = MAX_BATCH
opt = torch.optim.AdamW(enc.parameters(), lr=5e-5, weight_decay=1e-3)

def one_step():
    x = torch.randn(BS*HISTORY, 3, IMG, IMG, device='cuda')
    with torch.autocast('cuda', dtype=torch.bfloat16):
        out = enc(x).last_hidden_state[:, 0]
        loss = out.square().mean()
    opt.zero_grad(); loss.backward(); opt.step()

for _ in range(3):        # warmup — first steps are always slow
    one_step()
torch.cuda.synchronize()

N = 20
t0 = time.time()
for _ in range(N):
    one_step()
torch.cuda.synchronize()
sec_per_step = (time.time() - t0) / N

print(f'batch size      : {BS}')
print(f'sec / step      : {sec_per_step:.3f}')
print(f'steps / sec     : {1/sec_per_step:.2f}')
print()
print('NOTE: encoder only. Real steps also run the predictor + SIGReg,')
print('      so the true number is HIGHER. This is a floor.')

## 6 — The verdict

Multiply out. This is the number that decides Colab vs. paid compute.

In [ ]:
EPISODES        = 10_000
STEPS_PER_EP    = 92
FRAMESKIP       = 5
TRAIN_SPLIT     = 0.9

clips_per_ep    = STEPS_PER_EP // FRAMESKIP          # usable clips per episode
total_clips     = int(EPISODES * clips_per_ep * TRAIN_SPLIT)
steps_per_epoch = total_clips // BS

print(f'clips per episode : ~{clips_per_ep}')
print(f'total train clips : ~{total_clips:,}')
print(f'steps per epoch   : ~{steps_per_epoch:,}  (at batch {BS})')
print()

for epochs, label in [(10, 'paper Appendix E'), (100, 'repo config max_epochs')]:
    total_steps = steps_per_epoch * epochs
    hours = total_steps * sec_per_step / 3600
    print(f'{label:24s}: {epochs:3d} epochs = {total_steps:8,} steps = '
          f'{hours:6.1f} h  ({hours/4:.1f} sessions of 4h)')

print()
print('=' * 62)
print('READ THIS BEFORE DECIDING:')
print('=' * 62)
if BS < 128:
    print(f'* Batch {BS} != reference 128. This is a DEVIATION, not just slower.')
    print('  SIGReg is a statistical test computed ACROSS the batch, so a smaller')
    print('  batch makes it noisier. Gradient accumulation fixes the optimizer')
    print('  side but NOT this, because the statistic is still computed on')
    print(f'  {BS} samples at a time. Record this as a decision, do not absorb it.')
else:
    print('* Batch 128 fits — matches the reference config.')
print()
print('* Timing above is encoder-only, so real hours will be HIGHER.')
print('* Free Colab sessions are not guaranteed 4h; GPU type varies.')
print('* If total > ~15h, paid compute (~$10-20 on an A100) is likely cheaper')
print('  in hours-of-your-life than fighting timeouts across many sessions.')

## What to do with this

Record in the pre-registration, before any training run:

- the GPU Colab gave you, and its memory
- the largest batch that fit, and whether that deviates from 128
- measured seconds per step
- projected hours at 10 epochs and at 100
- **the epochs conflict**: repo config says 100, paper Appendix E says 10 for
  Two-Room. Decide which you're reproducing and write down why. The repo is the
  reference; the paper is the description. Where they conflict, follow the repo
  and record it.

Then decide: chunked Colab, or paid compute. Either is fine — but the choice
gets written down before the run, not justified after it.